# Resume-JD Match Scorer — Siamese BiLSTM

Classifies a (resume, job description) pair as Weak / Medium / Strong match.

Run all cells top to bottom. In Colab: Runtime -> Run all.

In [ ]:
!pip install -q tensorflow-cpu scikit-learn pandas matplotlib

## 1. Upload the raw dataset

Upload `train.jsonl` when prompted (Job-Description / Resume-matched / Resume-unmatched / Skills).

In [ ]:
from google.colab import files
uploaded = files.upload()  # select train.jsonl


## 2. Build the labelled dataset

Strong = JD + its true matched resume. Medium = JD + its own partial-skill (unmatched) resume. Weak = JD + a matched resume from a random different row.

In [ ]:
import json, csv, random
random.seed(42)

N_SOURCE_ROWS = 3000

def load_rows(path="train.jsonl", n=N_SOURCE_ROWS):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    random.shuffle(rows)
    return rows[:n]

def build_dataset(n_source_rows=N_SOURCE_ROWS, out_path="resume_jd_dataset.csv"):
    rows = load_rows(n=n_source_rows)
    pairs = []
    skill_counts = {}
    for i, row in enumerate(rows):
        jd = row["Job-Description"].strip()
        matched = row["Resume-matched"].strip()
        unmatched = row["Resume-unmatched"].strip()

        pairs.append((matched, jd, 2))
        pairs.append((unmatched, jd, 1))

        other_idx = random.choice([j for j in range(len(rows)) if j != i])
        other_resume = rows[other_idx]["Resume-matched"].strip()
        pairs.append((other_resume, jd, 0))

        for s in row.get("Skills", []):
            s = s.strip().lower()
            if s:
                skill_counts[s] = skill_counts.get(s, 0) + 1

    random.shuffle(pairs)
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["resume_text", "job_description", "match_label"])
        writer.writerows(pairs)

    top_skills = sorted(skill_counts, key=skill_counts.get, reverse=True)[:800]
    with open("skill_vocab.json", "w", encoding="utf-8") as f:
        json.dump(top_skills, f)

    counts = {0: 0, 1: 0, 2: 0}
    for _, _, label in pairs:
        counts[label] += 1
    print(f"wrote {len(pairs)} rows, class balance: {counts}")

build_dataset()


## 3. Preprocessing

Shared cleaning + tokenizer for both resume and JD text (same embedding table is used for both towers, so both need the same vocabulary).

In [ ]:
import re
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 140
VOCAB_SIZE = 15000

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def fit_tokenizer(resume_texts, jd_texts, vocab_size=VOCAB_SIZE):
    cleaned = [clean_text(t) for t in list(resume_texts) + list(jd_texts)]
    tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
    tokenizer.fit_on_texts(cleaned)
    return tokenizer

def texts_to_padded(texts, tokenizer, max_len=MAX_LEN):
    cleaned = [clean_text(t) for t in texts]
    sequences = tokenizer.texts_to_sequences(cleaned)
    return pad_sequences(sequences, maxlen=max_len, padding="post", truncating="post")

def prepare_dataset(df, tokenizer=None, max_len=MAX_LEN, vocab_size=VOCAB_SIZE):
    if tokenizer is None:
        tokenizer = fit_tokenizer(df["resume_text"], df["job_description"], vocab_size)
    resume_padded = texts_to_padded(df["resume_text"], tokenizer, max_len)
    jd_padded = texts_to_padded(df["job_description"], tokenizer, max_len)
    labels = np.array(df["match_label"])
    return resume_padded, jd_padded, labels, tokenizer

df = pd.read_csv("resume_jd_dataset.csv")
resume_padded, jd_padded, labels, tokenizer = prepare_dataset(df)
print(resume_padded.shape, jd_padded.shape, labels.shape)


## 4. Train / validation / test split

In [ ]:
from sklearn.model_selection import train_test_split

def split_data(resume_padded, jd_padded, labels, test_size=0.15, val_size=0.15, seed=42):
    r_temp, r_test, j_temp, j_test, y_temp, y_test = train_test_split(
        resume_padded, jd_padded, labels, test_size=test_size, stratify=labels, random_state=seed
    )
    val_fraction = val_size / (1 - test_size)
    r_train, r_val, j_train, j_val, y_train, y_val = train_test_split(
        r_temp, j_temp, y_temp, test_size=val_fraction, stratify=y_temp, random_state=seed
    )
    return (r_train, j_train, y_train), (r_val, j_val, y_val), (r_test, j_test, y_test)

train_set, val_set, test_set = split_data(resume_padded, jd_padded, labels)
r_train, j_train, y_train = train_set
r_val, j_val, y_val = val_set
r_test, j_test, y_test = test_set
print("train:", len(y_train), "val:", len(y_val), "test:", len(y_test))


## 5. Siamese BiLSTM model

Both towers (resume, JD) share the same embedding + BiLSTM. The pooled vectors are combined via concatenation, absolute difference, and elementwise product before the classifier head.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.saving import register_keras_serializable

@register_keras_serializable()
def abs_difference(tensors):
    u, v = tensors
    return tf.abs(u - v)

@register_keras_serializable()
def elementwise_product(tensors):
    u, v = tensors
    return u * v

def build_model(vocab_size, max_len, embed_dim=64, lstm_units=48,
                 dense_units=(128, 64), num_classes=3, dropout_rate=0.3):
    resume_input = layers.Input(shape=(max_len,), name="resume_input")
    jd_input = layers.Input(shape=(max_len,), name="jd_input")

    embedding = layers.Embedding(vocab_size, embed_dim, mask_zero=True, name="shared_embedding")
    bilstm = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=True), name="shared_bilstm")
    pool = layers.GlobalMaxPooling1D(name="shared_pool")

    def encode(x):
        x = embedding(x)
        x = bilstm(x)
        x = pool(x)
        return x

    u = encode(resume_input)
    v = encode(jd_input)

    diff = layers.Lambda(abs_difference, name="abs_diff")([u, v])
    prod = layers.Lambda(elementwise_product, name="elementwise_product_layer")([u, v])
    features = layers.Concatenate(name="comparison_features")([u, v, diff, prod])

    x = features
    for units in dense_units:
        x = layers.Dense(units, activation="relu")(x)
        x = layers.Dropout(dropout_rate)(x)

    output = layers.Dense(num_classes, activation="softmax", name="match_class")(x)
    return Model(inputs=[resume_input, jd_input], outputs=output, name="siamese_bilstm_resume_jd")

vocab_size = min(VOCAB_SIZE, len(tokenizer.word_index) + 1)
model = build_model(vocab_size=vocab_size, max_len=MAX_LEN)
model.summary()


## 6. Train

Class weights are applied so the model doesn't lean on the majority class.

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight

model.compile(optimizer=Adam(learning_rate=0.001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
class_weights_arr = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight = {i: w for i, w in enumerate(class_weights_arr)}

history = model.fit(
    [r_train, j_train], y_train,
    validation_data=([r_val, j_val], y_val),
    epochs=12,
    batch_size=64,
    callbacks=[early_stop],
    class_weight=class_weight,
    verbose=2,
)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history.history["loss"], label="train loss")
axes[0].plot(history.history["val_loss"], label="val loss")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(history.history["accuracy"], label="train acc")
axes[1].plot(history.history["val_accuracy"], label="val acc")
axes[1].set_title("Accuracy"); axes[1].legend()
plt.tight_layout()
plt.savefig("training_history.png")
plt.show()


## 7. Evaluate on the held-out test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

CLASS_NAMES = ["Weak", "Medium", "Strong"]
probs = model.predict([r_test, j_test], verbose=0)
preds = np.argmax(probs, axis=1)

print(classification_report(y_test, preds, target_names=CLASS_NAMES))
cm = confusion_matrix(y_test, preds)
print(cm)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(CLASS_NAMES); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion Matrix")
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                 color="white" if cm[i, j] > cm.max()/2 else "black")
fig.colorbar(im)
plt.tight_layout()
plt.savefig("confusion_matrix.png")
plt.show()


## 8. Save model + tokenizer

In [ ]:
import pickle

model.save("resume_jd_match_model.keras")
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
print("saved")


## 9. Predict on a new resume/JD pair

Includes a simple keyword-overlap explanation on top of the model's prediction (not part of the model itself).

In [ ]:
with open("skill_vocab.json") as f:
    SKILL_VOCAB = sorted(json.load(f), key=len, reverse=True)

def extract_skills(text):
    text = text.lower()
    found = set()
    for s in SKILL_VOCAB:
        # word-boundary match so short skills like r, c, ui don't match
        # inside unrelated words (e.g. r inside career)
        pattern = r"(?<![a-z0-9])" + re.escape(s) + r"(?![a-z0-9])"
        if re.search(pattern, text):
            found.add(s)
    return found

def predict_pair(resume_text, jd_text):
    r = texts_to_padded([resume_text], tokenizer, MAX_LEN)
    j = texts_to_padded([jd_text], tokenizer, MAX_LEN)
    p = model.predict([r, j], verbose=0)[0]
    pred_class = p.argmax()

    resume_skills = extract_skills(resume_text)
    jd_skills = extract_skills(jd_text)
    common = sorted(resume_skills & jd_skills)
    missing = sorted(jd_skills - resume_skills)

    print(f"Predicted class: {CLASS_NAMES[pred_class]} Match")
    print(f"Confidence: {p[pred_class]:.0%}")
    print(f"Common skills: {', '.join(common) or 'none found'}")
    print(f"Missing skills: {', '.join(missing) or 'none found'}")

# example - replace with your own resume/JD text
sample_resume = df.iloc[0]["resume_text"]
sample_jd = df.iloc[0]["job_description"]
predict_pair(sample_resume, sample_jd)
